In [57]:
import numpy as np
import matplotlib.pyplot as plt
import tqdm, utils, time
import matplotlib as mpl

P = 200
N0 = 500
N = 1000
T = 0
sigma = 0.2
lamb = 25
n_tasks = 5
n_paths = 10

plt.rcParams['figure.dpi'] = 200
mpl.rcParams['figure.figsize'] = [2, 1]
mpl.rcParams.update({'font.size': 6})
mpl.rcParams['lines.linewidth'] = 2

# all paths share the same teacher and test data
W_teach = np.random.normal(0, 1, (N0, N))
a_teach = np.random.normal(0, 1, (N, 1))
x_test = np.random.normal(0, 1, (P * n_tasks, N0))
y_test = utils.forward(x_test, W_teach, a_teach)

forgetting_matrix = np.zeros((n_paths, n_tasks + 1, n_tasks + 1))
generalization = np.zeros((n_paths, n_tasks + 1))
for k in tqdm.trange(n_paths):

    x1 = np.random.normal(0, 1, (P, N0))
    Y1 = utils.forward(x1, W_teach, a_teach)

    W1 = np.random.normal(0, sigma, (N0, N))
    X1 = utils.relu(N0 ** -0.5 * x1 @ W1)

    a1_samples, _ = utils.sample_at(target_t=Y1, features_t=X1,
                                    temp=T, std_t=sigma,
                                    num_samples=1, last_a=None, lambda_t=0)

    all_x = [x1]
    all_w = [W1]
    all_a = [a1_samples]
    all_y = [Y1]
    for i in range(n_tasks):
        new_x = all_x[-1] @ utils.get_rotation_mat(N0)
        all_x.append(new_x)
        new_y = utils.forward(new_x, W_teach, a_teach)
        # print(np.linalg.norm(new_y)**2 / P)
        all_y.append(new_y)
        new_w = np.random.normal(lamb / (lamb + sigma ** -2) * all_w[-1], np.sqrt((lamb + sigma ** -2) ** -1), (N0, N))
        # new_w = new_w / np.linalg.norm(new_w) * np.linalg.norm(W1)
        all_w.append(new_w)

        new_features = utils.relu(N0 ** -0.5 * new_x @ new_w)
        new_a, new_a_mean = utils.sample_at(last_a=all_a[-1], target_t=new_y,
                                            features_t=new_features, temp=T, std_t=sigma, lambda_t=lamb, num_samples=1)
        all_a.append(new_a_mean)

    for i in range(n_tasks + 1):
        for j in range(i + 1):
            forgetting_matrix[k, i, j] = utils.get_loss(inputs=all_x[j], weights=all_w[i], readout=all_a[i],
                                                        target=all_y[j])

    for i in range(n_tasks + 1):
        generalization[k, i] = utils.get_loss(inputs=x_test, weights=all_w[i], readout=all_a[i],
                                              target=y_test)

w_norms = np.array([np.linalg.norm(_w) ** 2 for _w in all_w])

plt.figure()
plt.plot(w_norms / N0 / N)
plt.tight_layout()
plt.show()

plt.figure()
plt.errorbar(np.arange(n_tasks + 1) + 1, generalization.mean(0), generalization.std(0),
             label='generalization error', ls='--', color='k')
for i in range(n_tasks + 1):
    plt.errorbar(np.arange(i, n_tasks + 1) + 1, forgetting_matrix[:, i:, i].mean(0), forgetting_matrix[:, i:, i].std(0))
# plt.ylim(-0.1, 1.5)
plt.ylabel('training loss on previous tasks')
plt.xlabel('after learning task #')
plt.title('Normalized MSE loss')
plt.grid()
plt.suptitle(f'P={P},N={N},N0={N0},T={T},lambda={lamb},sigma={sigma}')
plt.tight_layout()
plt.legend()
plt.show()

plt.figure()

delw = np.zeros(n_tasks + 1)
dela = np.zeros(n_tasks + 1)

theory_delw = (sigma ** -2 / (lamb + sigma ** -2)) ** 2 * sigma ** 2 + (lamb + sigma ** -2) ** -1
theory_dela = np.zeros(n_tasks + 1)
# theory_dela = sigma**-2 * ()
for i in range(n_tasks):
    delw[i + 1] = np.linalg.norm(all_w[i + 1] - all_w[i]) ** 2
    dela[i + 1] = np.linalg.norm(all_a[i + 1] - all_a[i]) ** 2

    # compute theory del a
    k_t_t_1 = utils.get_kernel(all_x[i + 1], all_x[i], all_w[i + 1], all_w[i])
    k_t_1 = utils.get_kernel(all_x[i], all_x[i], all_w[i], all_w[i])
    k_t = utils.get_kernel(all_x[i + 1], all_x[i + 1], all_w[i + 1], all_w[i + 1])
    vec = all_y[i + 1] - k_t_t_1 @ np.linalg.inv(k_t) @ all_y[i]
    theory_dela[i + 1] = float(vec.T @ np.linalg.inv(k_t) @ vec)

plt.figure()
plt.plot(delw / N0 / N, label='||del w||^2')
plt.axhline(theory_delw)

plt.plot(dela / N, label='||del a||^2')
plt.plot(theory_dela / N, label='||del a||^2 theory')
plt.title('per element change')
plt.legend()
plt.show()



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


100%|██████████| 20/20 [00:05<00:00,  3.56it/s]
